In [1]:
from dataclasses import dataclass
import logging
import os
import sys
from typing import Any, Dict, List, Tuple
from urllib.parse import urljoin

from sqlalchemy import create_engine, text, inspect
from sqlalchemy.engine.base import Engine
from dotenv import load_dotenv
import pandas as pd
import requests

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout,
    force=True,
)

logger = logging.getLogger('ETL')
logger.setLevel(logging.DEBUG)

load_dotenv()

True

In [2]:
API_BASE_URL = os.getenv('API_BASE_URL')
AUTH_ENDPOINT = '/auth'
GENRES_ENDPOINT = '/obras/v3/generos'
GENRE_MOVIES_ENDPOINT = '/obras/v3/generos/{idGenero}/filmes'
MOVIES_ENDPOINT = '/obras/v3/filmes/{idFilme}'
MOVIE_RATINGS_ENDPOINT = '/obras/v3/filmes/{idFilme}/avaliacoes'

In [3]:
def get_auth(url: str, username: str, password: str) -> str:
    """
    Returns a token from an basic authentication endpoint
    """
    response = requests.post(url, json={'username': username, 'password': password})

    try:
        response.raise_for_status()
        logger.debug('Successul authentication to API.')

    except Exception as error:
        logger.error('Unable to authenticate to API: %s.', error)
        raise

    return response.json().get('access_token')


def get_endpoint(url: str, headers: Dict[str, str], timeout: int = 5) -> Any:
    """
    Returns the result of a GET call to an endpoint
    """
    response = requests.get(url, headers=headers, timeout=timeout)

    try:
        response.raise_for_status()
        logger.debug('GET call executed with success to url %s.', url)

    except Exception as error:
        logger.error('Unable to authenticate to API: %s.', error)
        raise

    return response.json()

In [4]:
def extract() -> Tuple[List]:
    """
    Execute the Extract step of the ETL process
    """

    logger.info('-- STARTING EXTRACT STEP --')

    token = get_auth(
        urljoin(API_BASE_URL, AUTH_ENDPOINT),
        os.getenv('USERNAME'),
        os.getenv('PASSWORD'),
    )

    headers = {'Authorization': f'Bearer {token}', 'Accept': 'application/json'}

    genres = get_endpoint(urljoin(API_BASE_URL, GENRES_ENDPOINT), headers)
    logger.info('Downloaded %d objects from %s endpoint.', len(genres), GENRES_ENDPOINT)

    genres_movies = [
        movie
        for genre in genres
        for movie in get_endpoint(
            urljoin(
                API_BASE_URL, GENRE_MOVIES_ENDPOINT.format(idGenero=genre.get('id'))
            ),
            headers,
        )
    ]
    logger.info(
        'Downloaded %d objects from %s endpoint.',
        len(genres_movies),
        GENRE_MOVIES_ENDPOINT,
    )

    movies = [
        get_endpoint(
            urljoin(
                API_BASE_URL, MOVIES_ENDPOINT.format(idFilme=movie.get('id_movie'))
            ),
            headers,
        )
        for movie in genres_movies
    ]
    logger.info('Downloaded %d objects from %s endpoint.', len(movies), MOVIES_ENDPOINT)

    movies_ratings = [
        rating
        for movie in movies
        for rating in get_endpoint(
            urljoin(
                API_BASE_URL, MOVIE_RATINGS_ENDPOINT.format(idFilme=movie.get('id'))
            ),
            headers,
        )
    ]
    logger.info(
        'Downloaded %d objects from %s endpoint.',
        len(movies_ratings),
        MOVIE_RATINGS_ENDPOINT,
    )

    logger.info('-- EXTRACT STEP EXECUTED WITH SUCCESS --')

    return genres, movies, genres_movies, movies_ratings

In [5]:
def transform(
    genres: List, movies: List, genres_movies: List, movies_ratings: List
) -> pd.DataFrame:
    """
    Execute the Transform step of the ETL process
    """
    logger.info('-- STARTING TRANSFORM STEP --')

    df_genres = pd.DataFrame(genres)
    df_movies = pd.DataFrame(movies)
    df_genres_movies = pd.DataFrame(genres_movies)
    df_movies_ratings = pd.DataFrame(movies_ratings)

    df_aggregations = df_movies_ratings.groupby(['id_movie'], as_index=False).agg(
        qty_ratings=('rating', 'sum'),
        avg_rating=('rating', 'mean'),
        min_rating=('rating', 'min'),
        max_rating=('rating', 'max'),
    )

    df_exportation = (
        df_genres.merge(df_genres_movies, left_on='id', right_on='id_genre')
        .drop(columns=['id_x', 'id_y', 'id_genre'])
        .merge(df_movies, left_on='id_movie', right_on='id')
        .merge(df_aggregations, how='left', on='id_movie')
        .drop(columns=['id_movie', 'id'])
    )

    df_exportation['qty_ratings'] = df_exportation['qty_ratings'].fillna(0).astype(int)

    logger.info('-- TRANSFORM STEP EXECUTED WITH SUCCESS --')

    return df_exportation

In [6]:
@dataclass
class PostgresConnection:
    """
    Information required to connect to a Postgres instances
    """

    user: str
    password: str
    database: str
    host: str
    port: str

    @property
    def url(self) -> str:
        return (
            'postgresql+psycopg2://'
            f'{self.user}:{self.password}@{self.host}:{self.port}/{self.database}'
        )


def create_movie_table(engine: Engine):
    """
    Create the `movie` table if required
    """
    inspector = inspect(engine)

    if inspector.has_table('movie'):
        logger.info('Table "movie" already created. Skip creation.')
    else:
        with engine.begin() as connection:
            create_table_query = text("""
            CREATE TABLE IF NOT EXISTS movie (
                id INT UNIQUE NOT NULL,
                name VARCHAR(50) NOT NULL,
                genre VARCHAR(50) NOT NULL,
                release_data DATE NOT NULL,
                qty_ratings INT NOT NULL DEFAULT 0,
                avg_rating NUMERIC(3, 2),
                min_rating INT,
                max_rating INT
            );
            """)

            connection.execute(create_table_query)
            logger.info('Table "movies" created with success')


def load_movie_table(engine: Engine, df: pd.DataFrame):
    try:
        df.to_sql(name='movie', con=engine, if_exists='replace', index=False)
        logger.info('DataFrame successfully loaded into the movie table')

    except Exception as error:
        logger.error(f'Error loading data: {error}')
        raise

    finally:
        engine.dispose()


def count_rows_movie_table(engine: Engine) -> None:
    with engine.connect() as connection:
        count_query = text('SELECT COUNT(*) FROM movie;')
        result = connection.execute(count_query)
        total_lines = result.scalar()

    logger.debug('The "movies" table currently contains %s line(s)', total_lines)


def load(df: pd.DataFrame):
    """
    Execute the Load step from the ETL process.
    """
    logger.info('-- STARTING LOAD STEP --')

    conn = PostgresConnection(
        os.getenv('POSTGRES_USER'),
        os.getenv('POSTGRES_PASSWORD'),
        os.getenv('POSTGRES_DB'),
        os.getenv('POSTGRES_HOST'),
        os.getenv('POSTGRES_PORT'),
    )

    try:
        engine = create_engine(conn.url)
        logger.debug('SQLAlchemy: Successful connection to Postgres')

    except Exception as error:
        logger.error('SQLAlchemy: Error configuring the engine: %s', error)
        logger.debug('SQLAlchemy: Connection URI: %s', conn.url)
        raise

    try:
        create_movie_table(engine)
        count_rows_movie_table(engine)
        load_movie_table(engine, df)
        count_rows_movie_table(engine)

    finally:
        engine.dispose()

    logger.info('-- LOAD STEP EXECUTED WITH SUCCCESS --')

In [7]:
conn = PostgresConnection(
    os.getenv('POSTGRES_USER'),
    os.getenv('POSTGRES_PASSWORD'),
    os.getenv('POSTGRES_DB'),
    os.getenv('POSTGRES_HOST'),
    os.getenv('POSTGRES_PORT'),
)

engine = create_engine(conn.url)

with engine.begin() as connection:
    drop_table_query = text('DROP TABLE IF EXISTS movie;')
    connection.execute(drop_table_query)

logger.info('---- STARTING ETL PROCESS ----')
load(transform(*extract()))
logger.info('---- ETL PROCESS EXECUTED WITH SUCCESS ----')

conn = PostgresConnection(
    os.getenv('POSTGRES_USER'),
    os.getenv('POSTGRES_PASSWORD'),
    os.getenv('POSTGRES_DB'),
    os.getenv('POSTGRES_HOST'),
    os.getenv('POSTGRES_PORT'),
)

engine = create_engine(conn.url)

try:
    with engine.begin() as connection:
        query = text('SELECT * FROM movie;')
        df_movies = pd.read_sql_query(query, con=engine)
        print(df_movies.head())

finally:
    engine.dispose()

2026-06-24 16:50:51,768 - INFO - ---- STARTING ETL PROCESS ----
2026-06-24 16:50:51,769 - INFO - -- STARTING EXTRACT STEP --
2026-06-24 16:50:52,028 - DEBUG - Successul authentication to API.
2026-06-24 16:50:52,273 - DEBUG - GET call executed with success to url https://super-duper-spork-7r7vj59999pfrpwx-5000.app.github.dev/obras/v3/generos.
2026-06-24 16:50:52,276 - INFO - Downloaded 3 objects from /obras/v3/generos endpoint.
2026-06-24 16:50:52,486 - DEBUG - GET call executed with success to url https://super-duper-spork-7r7vj59999pfrpwx-5000.app.github.dev/obras/v3/generos/1/filmes.
2026-06-24 16:50:52,568 - DEBUG - GET call executed with success to url https://super-duper-spork-7r7vj59999pfrpwx-5000.app.github.dev/obras/v3/generos/2/filmes.
2026-06-24 16:50:52,883 - DEBUG - GET call executed with success to url https://super-duper-spork-7r7vj59999pfrpwx-5000.app.github.dev/obras/v3/generos/3/filmes.
2026-06-24 16:50:52,895 - INFO - Downloaded 4 objects from /obras/v3/generos/{idGe